In [ ]:
!pip install aiohttp nest_asyncio pandas tqdm > /dev/null

import requests, re, aiohttp, asyncio, nest_asyncio, pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

nest_asyncio.apply()

base_url = "https://www.promofarma.com.br"
categoria_url = f"{base_url}/medicamentos"
headers = {"User-Agent": "Mozilla/5.0"}

# ============================
# PASSO 1: Coletar links de produtos (todas as páginas)
# ============================
def coletar_links_categoria(url):
    links = []
    page = 1
    while True:
        print(f"🔎 Coletando página {page}...")
        resp = requests.get(f"{url}?page={page}", headers=headers)
        if resp.status_code != 200:
            print("❌ Erro HTTP", resp.status_code)
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        page_links = [a["href"] for a in soup.select("a[href*='/p']")]
        if not page_links:  # sem mais produtos
            break
        links.extend(page_links)
        page += 1
    return list(set(links))  # remove duplicados

produto_links = coletar_links_categoria(categoria_url)
print(f"✅ Total de produtos encontrados: {len(produto_links)}")

# ============================
# PASSO 2: Extrair productId real de cada página
# ============================
def extrair_product_id(produto_url):
    resp = requests.get(produto_url, headers=headers)
    if resp.status_code != 200:
        return None
    match = re.search(r'"productId":"(\d+)"', resp.text)
    if match:
        return match.group(1)
    return None

product_ids = []
for link in produto_links:
    url = base_url + link if link.startswith("/") else link
    pid = extrair_product_id(url)
    if pid:
        product_ids.append(pid)

print(f"🔑 Total de IDs encontrados: {len(product_ids)}")

# ============================
# PASSO 3: Montar URLs da API
# ============================
api_urls = [f"{base_url}/api/catalog_system/pub/products/search?fq=productId:{pid}" for pid in product_ids]
print("Exemplo de URL da API:", api_urls[0] if api_urls else "Nenhuma")

# ============================
# PASSO 4: Baixar JSONs em paralelo
# ============================
async def fetch(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                return None
            return await resp.json()
    except:
        return None

async def baixar_produtos(api_urls):
    produtos = []
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, url) for url in api_urls]
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            result = await f
            if result:
                produtos.extend(result)
    return produtos

produtos = asyncio.run(baixar_produtos(api_urls))
print(f"📦 Total de produtos coletados: {len(produtos)}")

# ============================
# PASSO 5: Transformar em CSV
# ============================
dados = []
for p in produtos:
    try:
        item = p["items"][0]
        oferta = item["sellers"][0]["commertialOffer"]
        dados.append({
            "id": p["productId"],
            "nome": p["productName"],
            "marca": p.get("brand", ""),
            "preco": oferta["Price"],
            "preco_de": oferta["ListPrice"],
            "estoque": oferta["AvailableQuantity"],
            "ean": item.get("ean", ""),
            "link": base_url + p["link"],
            "imagem": item["images"][0]["imageUrl"] if item["images"] else ""
        })
    except:
        continue

df = pd.DataFrame(dados)
df.to_csv("produtos.csv", index=False, encoding="utf-8-sig")
df.head()
